# G20b: 31B teacher scoring of the G20 student leaves (prompt_logprobs top-8)

Rerun of G20 stage 2 after its OOM (2 GiB log_softmax for a 2,048-token chunk); 512-token chunks, --enforce-eager, 0.72 GPU memory. Output `g20b/teacher_scores.npz`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

In [ ]:
import json, os, random, subprocess, sys, threading, time, traceback
from pathlib import Path
import requests, torch
W = Path('/kaggle/working/g20b'); W.mkdir(parents=True, exist_ok=True); CODE = W / 'code'; CODE.mkdir(exist_ok=True)
RESULTS = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
def save_results(): (W / 'g20b_summary.json').write_text(json.dumps(RESULTS, indent=1, default=str))
_gpus = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]
print('GPUs:', _gpus); assert len(_gpus) == 4 and all(c[0] >= 8 for _, c in _gpus), _gpus
BUDGET_MIN = 85; T_START = time.time()
def _watchdog():
    RESULTS['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
(CODE / 'opd_score.py').write_text("\"\"\"G20 (on-policy distillation, data stage): teacher scoring of student contexts.\n\nThe student (E4B) runs the real harness and the capture proxy records every request with exact prompt and completion token ids. For each context segment of the main\nagent (one per capture task_key; compaction starts a new one) the LEAF is the request with the longest prompt, and its sequence is prompt ids + completion ids: it\ncontains every model turn of that segment (as re-rendered in history, the same form the SFT sequences use). The teacher (31B, same tokenizer and chat template,\nverified 60/60 on real captures) scores each leaf in one prefill through vLLM's /v1/completions with `prompt_logprobs`: per position the log-probability of the\nactual token and the teacher's top-k tokens. Loss spans (the student's own tokens) are derived offline from the capture, as in scripts/g10/build_sft_general.py.\n\nOutput (npz, rewritten as it grows): ids int32 [N], leaf_offsets int64 [L+1], n_prompt int32 [L], act_lp float32 [N] (teacher log p of the actual token; NaN at\nposition 0), top_ids int32 [N,k], top_lp float16 [N,k] (-inf padding), plus meta.json (task_key, round, t0 of the leaf request, seconds).\n\"\"\"\n\nimport gzip\nimport json\nimport math\nimport time\n\nimport numpy as np\nimport requests\n\nMAIN = \"You are an expert autonomous software engineer\"\n\n\ndef _text(c):\n    if isinstance(c, list):\n        return \"\".join(p.get(\"text\", \"\") for p in c if isinstance(p, dict))\n    return c or \"\"\n\n\ndef read_records(path):\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n\n\ndef leaves(capture_path, round_prefix):\n    \"\"\"Main-agent leaves of the rounds whose tag starts with round_prefix, shortest first.\"\"\"\n    best = {}\n    for rec in read_records(capture_path):\n        if not str(rec.get(\"round\", \"\")).startswith(round_prefix):\n            continue\n        msgs = rec[\"request\"].get(\"messages\", [])\n        sysm = _text(next((m.get(\"content\") for m in msgs if m.get(\"role\") == \"system\"), \"\"))\n        if not sysm.startswith(MAIN):\n            continue\n        resp = rec.get(\"response\") or {}\n        p = resp.get(\"prompt_token_ids\")\n        c = ((resp.get(\"choices\") or [{}])[0]).get(\"token_ids\")\n        if not p or not c:\n            continue\n        k = rec[\"task_key\"]\n        if k not in best or len(p) > best[k][\"n_prompt\"]:\n            best[k] = {\"task_key\": k, \"round\": rec[\"round\"], \"t0\": rec.get(\"t0\"), \"n_prompt\": len(p), \"ids\": list(p) + list(c)}\n    return sorted(best.values(), key=lambda x: len(x[\"ids\"]))\n\n\ndef _tid(key):\n    s = str(key)\n    if s.startswith(\"token_id:\"):\n        s = s[len(\"token_id:\"):]\n    return int(s)\n\n\ndef _lp(v):\n    return float(v[\"logprob\"]) if isinstance(v, dict) else float(v)\n\n\ndef parse_prompt_logprobs(plp, ids, k):\n    \"\"\"vLLM prompt_logprobs (list, None at position 0, else {token_id: {logprob, rank, decoded_token}} with the actual token plus the top-k) -> arrays.\"\"\"\n    n = len(ids)\n    if len(plp) != n:\n        raise ValueError(f\"prompt_logprobs has {len(plp)} positions, sequence has {n}\")\n    act = np.full(n, np.nan, np.float32)\n    top_ids = np.full((n, k), -1, np.int32)\n    top_lp = np.full((n, k), -np.inf, np.float16)\n    for i, d in enumerate(plp):\n        if not d:\n            continue\n        items = sorted(((_tid(t), _lp(v)) for t, v in d.items()), key=lambda x: -x[1])\n        dd = dict(items)\n        if ids[i] not in dd:\n            raise ValueError(f\"actual token {ids[i]} missing at position {i}\")\n        act[i] = dd[ids[i]]\n        for j, (t, lp) in enumerate(items[:k]):\n            top_ids[i, j] = t\n            top_lp[i, j] = lp\n    return act, top_ids, top_lp\n\n\ndef score_leaf(url, model, ids, k, timeout=1800):\n    r = requests.post(url.rstrip(\"/\") + \"/v1/completions\",\n                      json={\"model\": model, \"prompt\": ids, \"max_tokens\": 1, \"temperature\": 0.0, \"prompt_logprobs\": k}, timeout=timeout)\n    if r.status_code != 200:\n        raise RuntimeError(f\"HTTP {r.status_code}: {r.text[:500]}\")\n    return r.json()[\"choices\"][0][\"prompt_logprobs\"]\n\n\nclass Store:\n    def __init__(self, k):\n        self.k, self.rows, self.meta = k, [], []\n\n    def add(self, leaf, act, top_ids, top_lp, seconds):\n        self.rows.append((np.asarray(leaf[\"ids\"], np.int32), leaf[\"n_prompt\"], act, top_ids, top_lp))\n        self.meta.append({\"task_key\": leaf[\"task_key\"], \"round\": leaf[\"round\"], \"t0\": leaf[\"t0\"], \"n_tokens\": len(leaf[\"ids\"]), \"n_prompt\": leaf[\"n_prompt\"], \"seconds\": round(seconds, 2)})\n\n    def save(self, npz_path, meta_path):\n        if not self.rows:\n            return\n        lens = [len(r[0]) for r in self.rows]\n        np.savez(npz_path, ids=np.concatenate([r[0] for r in self.rows]), leaf_offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),\n                 n_prompt=np.array([r[1] for r in self.rows], np.int32), act_lp=np.concatenate([r[2] for r in self.rows]),\n                 top_ids=np.concatenate([r[3] for r in self.rows]), top_lp=np.concatenate([r[4] for r in self.rows]))\n        with open(meta_path, \"w\", encoding=\"utf-8\") as f:\n            json.dump({\"k\": self.k, \"leaves\": self.meta}, f)\n\n\ndef score_all(leaves_, job, store, out_of_time, on_progress, workers=2, every=10):\n    \"\"\"Run job(leaf) -> (act, top_ids, top_lp, seconds) over leaves with `workers` requests in flight; stop submitting once out_of_time() is True.\n    Failed leaves are counted and their first errors kept; on_progress() is called every `every` finished leaves and at the end. Returns (n_ok, errors).\"\"\"\n    from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait\n    it, pending, errors, n_done, n_ok = iter(leaves_), {}, [], 0, 0\n    with ThreadPoolExecutor(workers) as ex:\n        def fill():\n            while len(pending) < workers and not out_of_time():\n                leaf = next(it, None)\n                if leaf is None:\n                    return\n                pending[ex.submit(job, leaf)] = leaf\n        fill()\n        while pending:\n            done, _ = wait(list(pending), return_when=FIRST_COMPLETED)\n            for f in done:\n                leaf = pending.pop(f)\n                try:\n                    act, top_ids, top_lp, sec = f.result()\n                    store.add(leaf, act, top_ids, top_lp, sec)\n                    n_ok += 1\n                except Exception as e:  # noqa: BLE001\n                    errors.append(f\"{leaf['task_key']}: {str(e)[:300]}\")\n                n_done += 1\n                if n_done % every == 0:\n                    on_progress()\n            fill()\n    on_progress()\n    return n_ok, errors\n\n\ndef summary(store):\n    \"\"\"Mean teacher log p of the actual tokens over the completion part of each leaf (a rough 'how much does the teacher agree with the student' number).\"\"\"\n    vals = []\n    for ids, n_prompt, act, _, _ in store.rows:\n        a = act[n_prompt:]\n        a = a[~np.isnan(a)]\n        if len(a):\n            vals.append(float(a.mean()))\n    return {\"leaves\": len(store.rows), \"tokens\": int(sum(len(r[0]) for r in store.rows)),\n            \"mean_teacher_lp_last_completion\": round(sum(vals) / len(vals), 4) if vals else None,\n            \"frac_last_completion_tokens_teacher_lp_below_ln0.1\": round(float(np.mean(np.concatenate([r[2][r[1]:] for r in store.rows]) < math.log(0.1))), 4) if store.rows else None}\n", encoding='utf-8'); sys.path.insert(0, str(CODE))
import opd_score as O
caps = [p for p in Path('/kaggle/input').rglob('capture.jsonl.gz') if p.parent.name == 'g20']
assert len(caps) == 1, caps
LEAVES = O.leaves(str(caps[0]), 'e1_')
RESULTS['leaves'] = {'n': len(LEAVES), 'tokens': sum(len(l['ids']) for l in LEAVES), 'max_len': max(len(l['ids']) for l in LEAVES)}
print('leaves:', RESULTS['leaves']); assert RESULTS['leaves']['n'] == 244 and RESULTS['leaves']['tokens'] == 2930160, RESULTS['leaves']
T_MODEL = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'); assert (T_MODEL / 'config.json').is_file(), T_MODEL   # path used by G20
K = 8
TCMD = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', str(T_MODEL), '--host', '127.0.0.1', '--port', '8010', '--max-model-len', '33000',
        '--dtype', 'bfloat16', '--gpu-memory-utilization', '0.72', '--tensor-parallel-size', '4', '--max-logprobs', '20', '--max-num-batched-tokens', '512',
        '--max-num-seqs', '8', '--enforce-eager']
RESULTS['teacher_cmd'] = ' '.join(TCMD); save_results(); print(RESULTS['teacher_cmd'], flush=True)
TURL = 'http://127.0.0.1:8010'
tproc = subprocess.Popen(TCMD, stdout=open(W / 'teacher_vllm.log', 'w'), stderr=subprocess.STDOUT)
t_boot = time.time(); ready = False
while time.time() - t_boot < 15 * 60 and tproc.poll() is None:
    try:
        if requests.get(TURL + '/health', timeout=5).status_code == 200: ready = True; break
    except Exception:
        pass
    time.sleep(10)
RESULTS['teacher_boot_s'] = round(time.time() - t_boot); RESULTS['teacher_ready'] = ready; save_results(); print('teacher ready:', ready, RESULTS['teacher_boot_s'], 's', flush=True)
STORE = O.Store(K)
def _save():
    STORE.save(W / 'teacher_scores.npz', W / 'teacher_meta.json'); RESULTS['teacher'] = O.summary(STORE); save_results()
if not ready:
    RESULTS['teacher_error'] = 'not ready: ' + (W / 'teacher_vllm.log').read_text(errors='ignore')[-3000:]
else:
    TMODEL = requests.get(TURL + '/v1/models', timeout=30).json()['data'][0]['id']
    def job(leaf):
        t = time.time()
        act, ti, tl = O.parse_prompt_logprobs(O.score_leaf(TURL, TMODEL, leaf['ids'], K), leaf['ids'], K)
        return act, ti, tl, time.time() - t
    try:
        a, ti, tl, sec = job(LEAVES[0]); STORE.add(LEAVES[0], a, ti, tl, sec); _save()
        RESULTS['teacher_probe'] = {'tokens': len(LEAVES[0]['ids']), 'seconds': round(sec, 1), 'tok_per_s': round(len(LEAVES[0]['ids']) / max(sec, 1e-6)),
                                    'act_lp_head': [float(x) for x in a[1:6]], 'top_ids_head': ti[1].tolist()}
        print('probe:', RESULTS['teacher_probe'], flush=True); save_results()
        rest = LEAVES[1:]; random.Random(20).shuffle(rest)
        n_ok, errs = O.score_all(rest, job, STORE, lambda: (time.time() - T_START) / 60 > BUDGET_MIN - 6, _save, workers=2, every=10)
        RESULTS['teacher_pass'] = {'scored': n_ok + 1, 'of': len(LEAVES), 'errors': len(errs), 'first_errors': errs[:3], 'minutes': round((time.time() - T_START) / 60, 1)}
    except Exception:
        RESULTS['teacher_error'] = traceback.format_exc()[-2500:]
        RESULTS['teacher_log_tail'] = (W / 'teacher_vllm.log').read_text(errors='ignore')[-2500:]
    _save()
tproc.terminate()
print('\n===== G20b SUMMARY =====\n', json.dumps({k: RESULTS.get(k) for k in ('leaves', 'teacher_boot_s', 'teacher_probe', 'teacher_pass', 'teacher', 'teacher_error')}, indent=1, default=str))
RESULTS['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); save_results(); _wd.cancel()
